# Phase 1: Multilingual Data Preprocessing & Exploratory Data Analysis (EDA)
**Project Title:** Multi-Script Language Detection for English, Hindi, and Kannada  
**Domain:** Natural Language Processing (NLP) & Multi-Class Text Classification  

### Phase 1 Objectives Condensed in this Notebook:
1. **Dataset Identification & Loading:** Ingest `language_dataset.csv` (1,050 samples)
2. **Exploratory Data Analysis (EDA):** Statistical profiling across sentence lengths, word counts, and writing scripts
3. **Data Visualizations:** Class distribution bar chart, character length histograms with KDE, word count boxplots
4. **Script-Preserving Preprocessing:** Unicode NFC normalization and regex filtering preserving Indic code points
5. **Stratified Splitting:** 80/20 train/test split maintaining exact 33.33% class balance
6. **Feature Extraction:** Sublinear TF-IDF character n-grams ($n \in [2, 5]$)
7. **Artifact Persistence:** Serialization of sparse matrices and fitted vectorizers


In [ ]:
import os
import re
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["font.size"] = 11

print("✔ Libraries imported successfully.")


## 1. Dataset Ingestion & Schema Inspection
We load `language_dataset.csv` containing balanced sentences across English (Latin), Hindi (Devanagari), and Kannada (Kannada script).


In [ ]:
# Locate dataset path across working directories
candidate_paths = [
    Path("data/language_dataset.csv"),
    Path("Phase-1/data/language_dataset.csv"),
    Path("../data/language_dataset.csv"),
    Path("language_dataset.csv")
]

dataset_path = None
for p in candidate_paths:
    if p.exists():
        dataset_path = p
        break

if not dataset_path:
    raise FileNotFoundError("language_dataset.csv not found in candidate paths!")

print(f"Loading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path)

print(f"Total Samples: {len(df)} | Columns: {list(df.columns)}")
print("\n--- First 5 Samples ---")
display(df.head())

print("\n--- Missing Values Check ---")
print(df.isnull().sum())


## 2. Exploratory Data Analysis (EDA) & Summary Statistics
We compute character lengths and word counts to uncover structural morphological differences among Latin, Devanagari, and Dravidian Kannada scripts.


In [ ]:
# Compute sentence metrics
df["char_length"] = df["Text"].astype(str).apply(len)
df["word_count"] = df["Text"].astype(str).apply(lambda s: len(s.split()))

print("--- Class Distribution ---")
class_dist = df["Language"].value_counts()
print(class_dist)

print("\n--- Character Length Statistics by Language ---")
len_stats = df.groupby("Language")["char_length"].agg(["mean", "std", "min", "max", "median"])
display(len_stats)

print("\n--- Word Count Statistics by Language ---")
word_stats = df.groupby("Language")["word_count"].agg(["mean", "std", "min", "max", "median"])
display(word_stats)


## 3. EDA Visualizations
We generate three comparative plots:
1. **Class Balance Bar Chart:** Verifies 350 / 350 / 350 uniform distribution.
2. **Character Length Distribution:** Visualizes sentence character spread with KDE curves.
3. **Word Count Distribution:** Highlights morphological differences (e.g. Kannada agglutination yielding fewer, longer words).


In [ ]:
palette = {"English": "#1f77b4", "Hindi": "#ff7f0e", "Kannada": "#2ca02c"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Class Distribution
sns.countplot(data=df, x="Language", hue="Language", palette=palette, legend=False, ax=axes[0])
axes[0].set_title("Class Balance Distribution (N=1,050)", fontweight="bold")
axes[0].set_xlabel("Language Class")
axes[0].set_ylabel("Number of Samples")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())} (33.3%)", 
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha="center", va="center", color="white", fontweight="bold")

# 2. Character Length Distribution
sns.histplot(data=df, x="char_length", hue="Language", palette=palette, kde=True, bins=25, alpha=0.4, ax=axes[1])
axes[1].set_title("Character Length Distribution (chars)", fontweight="bold")
axes[1].set_xlabel("Character Length")
axes[1].set_ylabel("Sample Count")

# 3. Word Count Boxplots
sns.boxplot(data=df, x="Language", y="word_count", hue="Language", palette=palette, legend=False, ax=axes[2])
axes[2].set_title("Word Count Distribution by Language", fontweight="bold")
axes[2].set_xlabel("Language Class")
axes[2].set_ylabel("Word Count")

plt.tight_layout()
plt.show()


## 4. Script-Preserving Preprocessing Pipeline
Standard English NLP packages (like Porter stemmers or ASCII regex filters) strip non-ASCII characters, which destroys Indic languages.  
Our custom preprocessing pipeline:
1. Performs **Unicode NFC Normalization** (`unicodedata.normalize('NFC', text)`).
2. Strips URLs and HTML markup.
3. Removes isolated numeric digits.
4. **Preserves Latin (`[a-zA-Z]`), Devanagari (`U+0900-U+097F`), and Kannada (`U+0C80-U+0CFF`) code points.**
5. Lowercases Latin without mutating Indic code points.
6. Normalizes redundant whitespaces.


In [ ]:
def clean_script_preserving(text: str) -> str:
    """Standardize text while strictly preserving Latin, Devanagari, and Kannada scripts."""
    if not isinstance(text, str):
        return ""
    # 1. NFC normalization
    text = unicodedata.normalize("NFC", text)
    # 2. Remove URLs and HTML tags
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    # 3. Remove digits
    text = re.sub(r"\d+", "", text)
    # 4. Filter characters strictly to target scripts and whitespace
    pattern = r"[^\sa-zA-Z\u0900-\u097F\u0C80-\u0CFF]"
    text = re.sub(pattern, " ", text)
    # 5. Lowercase Latin characters
    text = text.lower()
    # 6. Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply cleaning
df["cleaned_text"] = df["Text"].apply(clean_script_preserving)

print("--- Transformation Examples ---")
for i in [0, 350, 700]:
    row = df.iloc[i]
    print(f"[{row['Language']}] Raw:     {row['Text']}")
    print(f"[{row['Language']}] Cleaned: {row['cleaned_text']}\n")


## 5. Stratified Train / Test Splitting
We apply an **80/20 Stratified Train/Test Split** to maintain an exact $33.33\%$ representation for English, Hindi, and Kannada across both training and test partitions.


In [ ]:
# Target encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df["Language"])

print("Label Mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(f"  {idx} -> {label}")

# 80/20 Stratified split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["cleaned_text"],
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print(f"\nTraining set size: {len(X_train_raw)} samples (80%)")
print(f"Testing set size:  {len(X_test_raw)} samples (20%)")
print("Training class distribution:", np.bincount(y_train))
print("Testing class distribution: ", np.bincount(y_test))


## 6. Sublinear Character N-Gram Feature Extraction (TF-IDF)
Character $n$-grams (range $n \in [2, 5]$) provide sub-word script signatures that are robust against out-of-vocabulary terms and misspellings. Sublinear scaling applies $1 + \log(\text{tf})$ to prevent high-frequency characters from dominating the weights.


In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),      # Character n-grams from 2 to 5 chars
    sublinear_tf=True,       # Apply 1 + log(tf) scaling
    min_df=2,                # Prune extreme singletons
    max_features=50000
)

# Fit and transform
X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

vocab_size = len(vectorizer.vocabulary_)
print(f"✔ Vocabulary Extracted: {vocab_size} character n-gram features")
print(f"✔ X_train_tfidf Shape:  {X_train_tfidf.shape}")
print(f"✔ X_test_tfidf Shape:   {X_test_tfidf.shape}")
sparsity = 100.0 * (1.0 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]))
print(f"✔ Matrix Sparsity:      {sparsity:.2f}%")

# Sample features from each script
sample_features = list(vectorizer.get_feature_names_out())
latin_samples = [f for f in sample_features if re.search(r'[a-z]', f)][:5]
hindi_samples = [f for f in sample_features if re.search(r'[\u0900-\u097F]', f)][:5]
kannada_samples = [f for f in sample_features if re.search(r'[\u0C80-\u0CFF]', f)][:5]

print("\n--- Sample N-Gram Features ---")
print(f"Latin (English):   {latin_samples}")
print(f"Devanagari (Hindi):{hindi_samples}")
print(f"Kannada:           {kannada_samples}")


## 7. Sample Queries Preprocessing & Artifact Export
We test the pipeline on monolingual and code-mixed test sentences, and save preprocessed `.joblib` artifacts for Phase 2 model training.


In [ ]:
test_queries = [
    "Artificial intelligence is transforming science and technology.",
    "भारत की संस्कृति और इतिहास अत्यंत प्राचीन है।",
    "ಕನ್ನಡ ಸಾಹಿತ್ಯವು ಸಾವಿರಾರು ವರ್ಷಗಳ ಶ್ರೀಮಂತ ಇತಿಹಾಸವನ್ನು ಹೊಂದಿದೆ.",
    "data science and ತಂತ್ರಜ್ಞಾನ"  # Code-mixed query
]

print("--- Live Preprocessing & Vectorization Test ---")
for q in test_queries:
    cleaned = clean_script_preserving(q)
    vec = vectorizer.transform([cleaned])
    print(f"Original: {q}")
    print(f"Cleaned:  {cleaned}")
    print(f"Active N-Gram Features: {vec.nnz}\n")

# Export artifacts
output_dir = Path("preprocessed_artifacts")
output_dir.mkdir(exist_ok=True)

joblib.dump(X_train_tfidf, output_dir / "X_train_tfidf.joblib")
joblib.dump(X_test_tfidf, output_dir / "X_test_tfidf.joblib")
joblib.dump(y_train, output_dir / "y_train.joblib")
joblib.dump(y_test, output_dir / "y_test.joblib")
joblib.dump(vectorizer, output_dir / "tfidf_vectorizer.joblib")
joblib.dump(label_encoder, output_dir / "label_encoder.joblib")

print(f"✔ Preprocessed artifacts persisted in: {output_dir.resolve()}")
print("✔ Phase 1 data preprocessing & CSV usage notebook execution complete!")
